SVD is PCA.  

our data has good PCA, some components explain all variance.  
Use PCA principal components to initialise our atoms. Repeat them.   


Currently PCA overlaps. We want a non-overlapping solution, so simply repeat the pca components in our atoms as we increase them. The algorithm should converge hopefully.  


Note that the components we found for POI (2,4) or something have very less variance. There was definite pattern though, I need to decide if that is okay.  


It might be interesting to create a constrained solver where i can increase the number of atoms in a group (which ive assigned) one by one and see how many atoms maximally explain a group.  hmm hmm.   hmmmmmmmmmmmmm.   

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
import gc
import pickle

In [ ]:
MODEL_PATH = Path("../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../pt-to-api/data/first-input-tens.pt")
MAIN_OUT_DIR = (Path.cwd() / "mnist-patches-data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
PATCHES_DATA_ROOT = Path("/Users/hariomnarang/Desktop/personal/hiccup-ide/backend/notebooks/mnist-patches-data")
layer_name = "layers.2"
channel = 12
SHAPE = (8,9)
MAIN_DUMP_DIR = Path("runs-l2-o12")
MAIN_DUMP_DIR.mkdir(parents=True, exist_ok=True)


patches_ds = PATCHES_DATA_ROOT / layer_name / str(channel)
assert patches_ds.is_dir()

In [ ]:
def _mse(x, codes, comps):
    diff = x - (codes@comps)
    return (diff**2).mean()


def get_comp_scores(X, codes, components):
    main_mse = _mse(X, codes, components)
    scores = []
    for i in range(len(components)):
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(X, codes, comps)
        comp_score = new_mse - main_mse
        scores.append(comp_score)
    return scores

In [ ]:
def get_sampled_patches(num_samples=-1):
    patches = []
    for p in patches_ds.glob("*.pt"):
        patches.append(torch.load(p, weights_only=False, map_location="cpu").numpy())
    patches = np.concat(patches)
    samples_idxs = torch.randperm(patches.shape[0]).numpy()
    total = len(samples_idxs)
    to_extract = num_samples if num_samples != -1 else total

    samples = patches[samples_idxs[:to_extract]]
    return samples


In [ ]:
def save_weight_and_patches(out_dir, n_samples=-1):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)
    # only run once, uncomment if you wanna generate new samples
    to_save_patches = get_sampled_patches(n_samples)
    torch.save(to_save_patches, out_dir / "patches.pt")

    layer = model.get_submodule(f"{layer_name}")
    weight = layer.weight[channel].clone().detach()
    weight = weight.reshape(-1).numpy()
    torch.save(weight, out_dir / f"{layer_name}-{channel}.pt")

# save for uploading to colab
# save_weight_and_patches("./colab-assets-for-analysis")

In [ ]:
from pt_to_api.benchmark import MeanPerDimGlobalStdScaler, NormaliseStdScaler
from sklearn.decomposition import FastICA,PCA
from sklearn.preprocessing import StandardScaler
from pt_to_api.benchmark import train_x as TX
from dataclasses import dataclass
from pt_to_api import benchmark as B

layer = model.get_submodule(f"{layer_name}")
weight = layer.weight[channel].clone().detach()
weight = weight.reshape(-1).numpy()


patches = torch.load(MAIN_DUMP_DIR / "saved-patches-7000.pt", weights_only=False)
pw = patches * weight

scaler = NormaliseStdScaler().fit(pw)
scaled_pw = scaler.transform(pw)

In [ ]:
S([weight.reshape(8,9)])

In [ ]:
pca = PCA(n_components=50, whiten=False)
pca.fit(scaled_pw)
with np.printoptions(suppress=True, precision=5):
    print(pca.explained_variance_ratio_)
S([c.reshape(8,9) for c in pca.components_[:8]], 20, 8)
plt.show()

In [ ]:
import os
from contextlib import contextmanager
import time

@contextmanager
def training_alert(success_sound="Hero", error_sound="Basso", volume=5, repeats=5, gap=0.1):
    def play(sound):
        for _ in range(repeats):
            os.system(f"afplay /System/Library/Sounds/{sound}.aiff -v {volume}")
            # time.sleep(gap)
    try:
        yield
        # play(success_sound)
    finally:
        play(error_sound)

# Training

## Warmup

In [ ]:
# i can do 16, no need for 21
import pickle
code = "warmup"
dump_dir = MAIN_DUMP_DIR / code
dump_dir.mkdir(parents=True, exist_ok=True)
EPOCHS, BASELINE_EPOCHS = 1500, 1000

for comp in range(2, 16):
    p = dump_dir / f"r{comp}.pkl"
    if p.exists():
        continue
    print("#############", comp)
    run = TX.train(
        scaled_pw, 
        comp, 
        1e-2, 
        device="mps", 
        epochs=EPOCHS, 
        baseline_epochs=BASELINE_EPOCHS, 
        batch_size=128, 
        use_ln_term=False, 
        init_strategy=B.WarmupInitStrategy(700),
    )
    with open(p, "wb") as f:
        pickle.dump(run, f)
    gc.collect()
        # sigma_eps_override=1e-2,
        # recon_err_schedule=TX.CosineAnnealReconError(1000)

## Std

In [ ]:
# i can do 16, no need for 21
with training_alert():
    code = "standard"
    dump_dir = MAIN_DUMP_DIR / code
    dump_dir.mkdir(parents=True, exist_ok=True)
    EPOCHS, BASELINE_EPOCHS = 1500, 1000

    for comp in range(2, 16):
        p = dump_dir / f"r{comp}.pkl"
        if p.exists():
            continue
        print("#############", comp)
        run = TX.train(
            scaled_pw, 
            comp, 
            1e-2, 
            device="mps", 
            epochs=EPOCHS, 
            baseline_epochs=BASELINE_EPOCHS, 
            batch_size=128, 
            use_ln_term=False, 
        )
        with open(p, "wb") as f:
            pickle.dump(run, f)
        del run
        collected = gc.collect()
        print("collected bytes", collected)
        torch.mps.empty_cache()

## cosine anneal

In [ ]:
S([scaled_pw[0].reshape(SHAPE), scaler.inverse_transform(scaled_pw[0]).reshape(SHAPE)], viztype="local")

In [ ]:
np.sqrt(0.05653649941086769)

In [ ]:
import pickle

# found in analysis copied manually
n_by_best_seed = {8: np.int64(1),
 9: np.int64(2),
 10: np.int64(0),
 11: np.int64(2),
 12: np.int64(2),
 13: np.int64(0),
 14: np.int64(0),
 15: np.int64(2)}

with training_alert(repeats=1):
    code = "cosine-anneal-normalize-std-test"


    dump_dir = MAIN_DUMP_DIR / code
    dump_dir.mkdir(parents=True, exist_ok=True)
    EPOCHS, BASELINE_EPOCHS = 1500, 1000

    for comp in range(11, 12):
        # we now want to do across seeds
        # 5 seeds
        # retrain on min loss idxs, but with log term

        src_path = dump_dir / "comps" / f"{comp}" / f"seed_{n_by_best_seed[comp]}.pkl"

        with open(src_path, "rb") as f:
            print("readdddddd", src_path)
            run = pickle.load(f)
        initialised_model = run.model

        p = dump_dir / "comps" / f"{comp}" / f"best_seed_with_log.pkl"
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            continue
        print("#############", comp, "target", p)
        run = TX.train(
            scaled_pw, 
            comp, 
            1e-2,
            device="mps", 
            epochs=EPOCHS, 
            baseline_epochs=BASELINE_EPOCHS, 
            batch_size=64, 
            use_ln_term=True,
            recon_err_schedule=TX.CosineAnnealReconError(1000),
            initialised_model=initialised_model,
            # sigma_eps_override=0.23777405117225825,
        )
        with open(p, "wb") as f:
            pickle.dump(run, f)
        del run

        gc.collect()
        torch.mps.empty_cache()

In [ ]:
p = dump_dir / "comps" / f"11" / f"best_seed_with_log.pkl"
with open(p, "rb") as f:
    run = pickle.load(f)

In [ ]:
# shits happenin here lol
S([c.reshape(SHAPE) for c in run.components], (20,3), len(run.components), viztype="local")

In [ ]:
import pickle

with training_alert():
    code = "cosine-anneal-normalize-std-test"
    dump_dir = MAIN_DUMP_DIR / code
    dump_dir.mkdir(parents=True, exist_ok=True)
    EPOCHS, BASELINE_EPOCHS = 1500, 1000

    for comp in range(8, 16):
        # we now want to do across seeds
        # 5 seeds
        for seed in range(3):
            torch.manual_seed(seed)
            np.random.seed(seed)
            p = dump_dir / "comps" / f"{comp}" / f"seed_{seed}.pkl"
            p.parent.mkdir(parents=True, exist_ok=True)
            if p.exists():
                continue
            print("#############", comp, "target", p)
            run = TX.train(
                scaled_pw, 
                comp, 
                1e-2,
                device="mps", 
                epochs=EPOCHS, 
                baseline_epochs=BASELINE_EPOCHS, 
                batch_size=64, 
                use_ln_term=False,
                recon_err_schedule=TX.CosineAnnealReconError(1000),
            )
            with open(p, "wb") as f:
                pickle.dump(run, f)
            del run

            gc.collect()
            torch.mps.empty_cache()

In [ ]:
with open(p, "rb") as f:
    run = pickle.load(f)

In [ ]:
S([c.reshape(SHAPE) for c in run.components], (20,5), 6)
plt.show()

In [ ]:
# run.recon[0]
recon, codes, _ = run.model(torch.tensor(scaled_pw))
codes = codes.detach().numpy()
recon = recon.detach().numpy()

In [ ]:
S([recon[0].reshape(SHAPE), scaled_pw[0].reshape(SHAPE)])

In [ ]:
comps = []
for i in range(len(codes[0])):
    comp = codes[0][i] * run.components[i]
    comps.append(comp.reshape(SHAPE))

In [ ]:
S(comps, (20,5), 6, ax_titles=[f"{c:.4f}" for c in codes[0]])
plt.show()

In [ ]:
S([scaler.inverse_transform(recon[0]).reshape(SHAPE)])

## Cosine anneal hyperparameter with CosineAnnealWarmRestart schedule with SGD

Prelimnary results weren't the best and it takes too long, skipping for now. might come back again later.  

In [ ]:
# i can do 16, no need for 21
import pickle

with training_alert():
    code = "cosine-anneal-and-sgd-with-warm-restarts"
    dump_dir = MAIN_DUMP_DIR / code
    dump_dir.mkdir(parents=True, exist_ok=True)
    EPOCHS, BASELINE_EPOCHS = 4000, 1000

    for comp in range(2, 16):
        p = dump_dir / f"r{comp}.pkl"
        if p.exists():
            continue
        print("#############", comp)
        run = TX.train(
            scaled_pw, 
            comp, 
            1e-2, 
            device="mps", 
            epochs=EPOCHS, 
            baseline_epochs=BASELINE_EPOCHS, 
            batch_size=128, 
            use_ln_term=False, 
            recon_err_schedule=TX.CosineAnnealReconError(1000),
            optim_type=TX.SGDOptimType(),
            sched_type=TX.CosineAnnealingWithWarmRestartsSchedType(200, 2, 1e-5),
        )
        with open(p, "wb") as f:
            pickle.dump(run, f)
        del run
        gc.collect()
        torch.mps.empty_cache()

In [ ]:
with open(dump_dir / "r15.pkl", "rb") as f:
    run = pickle.load(f)

In [ ]:
run.loss

## Analysis

In [ ]:
import pickle

c2r = {}

for i in range(2, 21):
    d = dump_dir /f"r{i}.pkl"
    with open(d, "rb") as f:
        run = pickle.load(f)
        c2r[i] = run

In [ ]:
plt.plot(list(range(2,21)), [c2r[i].loss for i in range(2,21)])
plt.show()

interpreting the correct components is still a problem right now.   
Decomposition is quite ruthless and seems arbitrary many times.  

This seems to be because it is finding different permutations of dimensions getting clubbed together at different runs.  
The silver lining is that it is still a bit interpretable.   

Consider the row 1 and 2.  

We see an increase in the loss going from row 1 -> 2.  
We can attribute it to the second component in row 2 having the merge of the left side of the components of row 1 (the top left square formed by 4 pixels).  
They explain the maximum variance in this dataset (they light up a lot, not necessarily together).  

So technically, row 2 does not have the ideal solution, so the first thing we want is "ideal" solutions for each row.   
How can we do that? should we decrease sigma_eps?   
We can do many runs with different seeds. and find the best components, although it is quite complicated to say "what is best". the easiest is using MSE, i guess using MSE is a good proxy. Yea, tis fine.   

Strategy 1:
- find ideal configuration for each n_components run using MSE. Run with multiple seeds. Hope that we find the best configuration. This should smoothen the graph.   
This is very inefficient though.  


The main problem is that the weight loss is kicking in too early? and it is giving up on reconstruction.   
We need a better schedule for prioritising reconstruction. Simply increasing sigma_eps is also not very productive.   


What we can do is decay the strength of the reconstruction maybe. It might be useful to test with warmup components also.   
Reconstruction can begin with 100x eps and go down fast, exponential decay is preferred. To the expected value ofcourse.  
Might be useful to do cosine annealing. The first thing we do is try with warmups first.   

And bc, save the inputs also.  
This will again take time though.  

But oh well.  


What do i need to do?
- Save the inputs
- Save the runs, do the warmup ones first
- Later we'll also see if we can do the older run again, with the new samples considering that ive lost them, and cant see the score.  
- I need the score and the MSE

In [ ]:
import math

for i in range(2,21):
    cols = min(i, 10)
    rows = math.ceil(i / cols)
    rsize = 2*rows
    csize = 20

    print("shape", i)
    S([c.reshape(SHAPE) for c in c2r[i].components], (csize, rsize), cols, viztype="local", suptitle=f"loss: {c2r[i].loss}")
    plt.show()

In [ ]:
EPOCHS, BASELINE_EPOCHS = 1500, 1000

for comp in range(2, 21):
    run = TX.train(scaled_pw, comp, 1e-2, device="mps", epochs=EPOCHS, baseline_epochs=BASELINE_EPOCHS, batch_size=128, use_ln_term=False)
    with open(dump_dir / f"r{comp}.pkl", "wb") as f:
        pickle.dump(run, f)
    gc.collect()

In [ ]:
# dump_dir = Path("./pkl-runs-of-comps-svd")
# dump_dir.mkdir(parents=True, exist_ok=True)
# EPOCHS, BASELINE_EPOCHS = 1500, 1000

# for comp in range(2, 21):
#     run = TX.train(
#         scaled_pw, comp, 1e-2, device="mps", epochs=EPOCHS, baseline_epochs=BASELINE_EPOCHS, batch_size=128, use_ln_term=False, init_strategy=B.SvdInitStrategy()
#     )
#     with open(dump_dir / f"r{comp}.pkl", "wb") as f:
#         pickle.dump(run, f)
#     gc.collect()

In [ ]:
with open(dump_dir / "r2.pkl", "rb") as f:
    run = pickle.load(f)

In [ ]:
# scores and components for comps=20
scores = get_comp_scores(scaled_pw, run.codes, run.components)
scores = (scores / np.sum(scores))*100

S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components), viztype="local", ax_titles=[f"{s:.2f}" for s in scores])
plt.show()
# S([c.reshape(SHAPE) for c in run.components])

In [ ]:
run = TX.train(scaled_pw, 10, 1e-2, device="mps", epochs=1000, baseline_epochs=1000, batch_size=128, use_ln_term=False)
S([c.reshape(SHAPE) for c in run.components], 10, len(run.components), viztype="local")
plt.show()

In [ ]:
run.loss

In [ ]:
# can i jsut, start with the comps found in max, then remove the lower ones one by one?
# what happens then? If i remove a component (the lowest score one, we have an inc in the loss)
# the loss continuously increases, we simply find the point where it starts
# well this is essentially the elbow only, so starting from below is fine. Imma feelin quite dumb hmmm
# the problem is having PCA components which are too weak for unexplained variance in the lower number of components

In [ ]:
# scores and components for comps=20
scores = get_comp_scores(scaled_pw, run2.codes, run2.components)
scores = (scores / np.sum(scores))*100

S([c.reshape(SHAPE) for c in run2.components], (20,5), len(run2.components), viztype="local", ax_titles=[f"{s:.2f}" for s in scores])
plt.show()

In [ ]:
# scores and components for comps=20
scores = get_comp_scores(scaled_pw, run.codes, run.components)
scores = (scores / np.sum(scores))*100

S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components), viztype="local", ax_titles=[f"{s:.2f}" for s in scores])
plt.show()

In [ ]:
# scores and components for comps=20
scores = get_comp_scores(scaled_pw, run.codes, run.components)
scores = (scores / np.sum(scores))*100

S([c.reshape(SHAPE) for c in run.components], (20,5), len(run.components) // 2, viztype="local", ax_titles=[f"{s:.2f}" for s in scores])
plt.show()